In [4]:
import numpy as np
import pandas as pd
import yfinance as yf


def fetch_market_data(tickers, start_date, end_date):
    print(f"Data for {tickers}")
    data = yf.download(tickers, start=start_date, end=end_date)
    return data


In [5]:
def process_risk_and_liquidity(data, ticker):
    """Calculate Log Returns, Rolling Volatility, Amihud Liquidity Ratio, and Parametric VaR."""

    df = pd.DataFrame()
    df["Close"] = data["Close"][ticker]
    df["Volume"] = data["Volume"][ticker]


    df = df.dropna().copy()

    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

    #Daily USD Dollar Volume (Price * Volume)
    df["Dollar_Volume"] = df["Close"] * df["Volume"]

    #Amihud Liquidity Ratio = |Return| / Dollar Volume (Scaled by 1e6 for readability)
    #Measures price impact per $1M traded
    df["Amihud_Liquidity_Impact"] = (
        df["Log_Return"].abs() / df["Dollar_Volume"]
    ) * 1e9

    #Rolling Realized Volatility (Annualized 10-day & 30-day windows)
    #Trading days = 252
    df["Vol_10D_Ann"] = (
        df["Log_Return"].rolling(window=10).std() * np.sqrt(252) * 100
    )
    df["Vol_30D_Ann"] = (
        df["Log_Return"].rolling(window=30).std() * np.sqrt(252) * 100
    )

    portfolio_value = 100000
    rolling_std = df["Log_Return"].rolling(window=30).std()

    df["VaR_95_USD"] = portfolio_value * (1.645 * rolling_std)
    df["VaR_99_USD"] = portfolio_value * (2.326 * rolling_std)

    df = df.dropna().round(8)
    df["Ticker"] = ticker

    return df


In [6]:
if __name__ == "__main__":
    TICKERS = ["2800.HK", "HKD=X"]
    START_DATE = "2024-01-01"
    END_DATE = "2026-08-01"

    raw_data = fetch_market_data(TICKERS, START_DATE, END_DATE)

    # Process metrics for HKEX ETF
    hkex_df = process_risk_and_liquidity(raw_data, "2800.HK")

    # Save to CSV
    output_filename = "lab49_capital_markets_data.csv"
    hkex_df.to_csv(output_filename)

    print(f"Data saved to '{output_filename}'")
    print(
        hkex_df[
            [
                "Close",
                "Log_Return",
                "Vol_30D_Ann",
                "Amihud_Liquidity_Impact",
                "VaR_95_USD",
            ]
        ].head()
    )

Data for ['2800.HK', 'HKD=X']


/tmp/ipykernel_515/3645391586.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  2 of 2 completed

Data saved to 'lab49_capital_markets_data.csv'
                Close  Log_Return  Vol_30D_Ann  Amihud_Liquidity_Impact  \
Date                                                                      
2024-02-15  14.908734    0.003738    27.389647                 0.002242   
2024-02-16  15.298141    0.025784    28.426622                 0.004967   
2024-02-19  15.103438   -0.012809    28.639589                 0.003330   
2024-02-20  15.186880    0.005509    28.627474                 0.001961   
2024-02-21  15.437215    0.016349    28.464229                 0.002924   

             VaR_95_USD  
Date                     
2024-02-15  2838.259282  
2024-02-16  2945.716096  
2024-02-19  2967.784817  
2024-02-20  2966.529450  
2024-02-21  2949.613129  
